# HydroSense-Kenya – Level 6: AI-Assisted Programming, Testing, Reproducibility & Presentation
**ICS 2207 Scientific Computing | February – May 2026**
---

## 1. AI Use Documentation
All AI-assisted work is logged in `AI_USE_LOG.md`. This cell summarises the key entries.

In [ ]:
ai_log = [
    {
        "Level": "L1", "Task": "Problem statement phrasing",
        "Prompt": "Draft a 500-word scientific problem statement for smart irrigation in Kenya",
        "Accepted": "Partly", "Modified": "Added specific Kenya Vision 2030 references and local crop context",
        "Validation": "Cross-checked against project brief requirements"
    },
    {
        "Level": "L2", "Task": "Timing comparison template",
        "Prompt": "Show me how to benchmark a Python loop vs NumPy using time.perf_counter",
        "Accepted": "Yes", "Modified": "Adapted N_REPS and units to microseconds for clarity",
        "Validation": "Ran multiple times; variance < 5%"
    },
    {
        "Level": "L3", "Task": "Bisection convergence plot",
        "Prompt": "Generate code to plot convergence error on a log scale for root-finding methods",
        "Accepted": "Partly", "Modified": "Added Secant method comparison; fixed axis labels",
        "Validation": "Verified root x=2 for f(x)=x^2-4 matches known exact value"
    },
    {
        "Level": "L5", "Task": "Monte Carlo rainfall generation",
        "Prompt": "Write a function to generate 1000 rainfall scenarios using numpy random",
        "Accepted": "Partly", "Modified": "Changed normal to clipped-normal (no negative rainfall); added seed param",
        "Validation": "np.all(result >= 0) = True; mean/std match input parameters within tolerance"
    },
    {
        "Level": "L6", "Task": "pytest test case generation",
        "Prompt": "Generate pytest test cases for a bisection root-finding function",
        "Accepted": "Partly", "Modified": "Added irrigation-domain test; tightened tolerances to 1e-5",
        "Validation": "All tests pass against manual Gaussian elimination verification"
    }
]

import pandas as pd
df_log = pd.DataFrame(ai_log)
print("=== AI Use Log Summary ===")
print(df_log[['Level','Task','Accepted','Modified']].to_string(index=False))

## 2. Automated Tests

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', '../tests/', '-v', '--tb=short'],
    capture_output=True, text=True, cwd='../src'
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

## 3. Full Integration – Reproduce All Key Results

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from data_cleaning import load_datasets, clean_weather, clean_soil
from numerical_methods import bisection, trapezoidal_rule, gaussian_elimination
from simulation import runge_kutta_simulation, monte_carlo_rainfall, summarise_monte_carlo
from optimization import minimise_water_use

plt.rcParams.update({'figure.dpi':120,'axes.grid':True,'grid.alpha':0.3,
                     'axes.spines.top':False,'axes.spines.right':False,'font.size':11})

# ── Load & clean ──────────────────────────────────────────────────────────────
weather_raw, soil_raw, params = load_datasets('../data/raw')
weather = clean_weather(weather_raw.copy())
soil    = clean_soil(soil_raw.copy())
weather['et_mm'] = np.maximum(0, 0.12*weather['temperature_c'] + 0.35*weather['wind_speed_mps']
                               + 2.4*weather['solar_index'] - 0.025*weather['humidity_pct'])
rain_arr = weather['rainfall_mm'].fillna(0).values
et_arr   = weather['et_mm'].values
print("✅ Data loaded and cleaned.")

# ── Numerical methods ─────────────────────────────────────────────────────────
def f_irr(I, S_t=27.9, R=0, ET=4.5, S_tgt=33.0):
    return S_t + 0.1*(R+I) - ET - S_tgt

r = bisection(f_irr, 0, 200, tol=1e-6)
print(f"✅ Root finding: I = {r['root']:.4f} mm in {r['iterations']} iterations.")

deficit = np.maximum(0, et_arr - rain_arr*0.1)
total_deficit = trapezoidal_rule(deficit, dx=1.0)
print(f"✅ Integration: 30-day cumulative deficit = {total_deficit:.4f} mm-days.")

A = np.array([[1.,1.,1.],[1.,0.,-1.5],[0.,1.,-1.2]])
b = np.array([60.,0.,0.])
x = gaussian_elimination(A, b)
print(f"✅ Linear system: I_A={x[0]:.2f}, I_B={x[1]:.2f}, I_C={x[2]:.2f} mm.")

# ── Simulation & Monte Carlo ──────────────────────────────────────────────────
zc = params[params['zone_id']=='Zone_C'].iloc[0]
S0_C = soil[soil['zone_id']=='Zone_C'].sort_values('timestamp')['soil_moisture_pct'].iloc[0]
irr_zero = np.zeros(30)
rk_S = runge_kutta_simulation(S0_C, rain_arr, et_arr, irr_zero, zc['field_capacity_pct'],
                               zc['drainage_coefficient'], zc['min_moisture_pct'])
print(f"✅ RK4 simulation: final moisture = {rk_S[-1]:.4f}% | stress days = {np.sum(rk_S[1:] < zc['min_moisture_pct'])}")

mc = monte_carlo_rainfall(rain_arr.mean(), rain_arr.std(), n_scenarios=1000, n_days=30, seed=42)
mc_m = np.array([runge_kutta_simulation(S0_C, mc[i], et_arr, irr_zero,
                  zc['field_capacity_pct'], zc['drainage_coefficient'], zc['min_moisture_pct'])
                  for i in range(1000)])
shortage_prob = np.mean(mc_m[:,-1] < zc['min_moisture_pct']) * 100
print(f"✅ Monte Carlo (1000): shortage probability = {shortage_prob:.1f}%")

# ── Optimization ──────────────────────────────────────────────────────────────
opt = minimise_water_use(S0_C, rain_arr, et_arr, zc['field_capacity_pct'],
                          zc['drainage_coefficient'], zc['min_moisture_pct'],
                          zc['target_moisture_pct'], n_iter=200, lr=0.05)
print(f"✅ Optimization: total water = {opt['total_water_used_mm']:.2f} mm | stress days = {opt['stress_days']}")
print("\nAll components verified and reproducible ✅")

## 4. Final Summary Dashboard

In [ ]:
ZONE_COLORS = {'Zone_A':'#e07b39','Zone_B':'#3a7ebf','Zone_C':'#5aab61'}
ZONE_CROPS  = {'Zone_A':'Tomato','Zone_B':'Kale','Zone_C':'Maize'}

fig = plt.figure(figsize=(16, 11))
fig.suptitle('HydroSense-Kenya – Final Integration Dashboard\nMarch 2026 | ICS 2207 Scientific Computing',
             fontsize=14, fontweight='bold', y=0.99)

# Panel 1: Rainfall & ET
ax1 = fig.add_subplot(3, 3, 1)
ax1b = ax1.twinx()
ax1.bar(weather['date'], weather['rainfall_mm'], color='steelblue', alpha=0.6)
ax1b.plot(weather['date'], weather['et_mm'], 'r-', lw=1.5)
ax1.set_title('Rainfall & ET', fontweight='bold', fontsize=10)
ax1.set_ylabel('Rain (mm)', fontsize=8); ax1b.set_ylabel('ET (mm/d)', fontsize=8, color='red')
ax1.tick_params(axis='x', rotation=30, labelsize=7)

# Panel 2: Soil moisture all zones
ax2 = fig.add_subplot(3, 3, 2)
for z, grp in soil.groupby('zone_id'):
    grp = grp.sort_values('timestamp')
    ax2.plot(grp['timestamp'], grp['soil_moisture_pct'], color=ZONE_COLORS[z], lw=1.5, label=z)
for _, row in params.iterrows():
    ax2.axhline(row['min_moisture_pct'], color=ZONE_COLORS[row['zone_id']], ls=':', lw=0.8)
ax2.set_title('Soil Moisture by Zone', fontweight='bold', fontsize=10)
ax2.set_ylabel('SM (%)', fontsize=8); ax2.legend(fontsize=7)
ax2.tick_params(axis='x', rotation=30, labelsize=7)

# Panel 3: Boxplots
ax3 = fig.add_subplot(3, 3, 3)
data_by_zone = [soil[soil['zone_id']==z]['soil_moisture_pct'].dropna().values for z in ['Zone_A','Zone_B','Zone_C']]
bp = ax3.boxplot(data_by_zone, patch_artist=True, labels=['A\nTomato','B\nKale','C\nMaize'])
for patch, color in zip(bp['boxes'], ['#e07b39','#3a7ebf','#5aab61']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax3.set_title('SM Distribution', fontweight='bold', fontsize=10); ax3.set_ylabel('SM (%)', fontsize=8)

# Panel 4: Root finding convergence
ax4 = fig.add_subplot(3, 3, 4)
from numerical_methods import bisection, newton_raphson, secant
f  = lambda I: 27.9 + 0.1*I - 4.5 - 33.0
df = lambda I: 0.1
rb = bisection(f, 0, 200); rn = newton_raphson(f, df, 30.0); rs = secant(f, 20.0, 50.0)
for name, r, col in [('Bisection',rb,'steelblue'),('N-R',rn,'firebrick'),('Secant',rs,'#5aab61')]:
    if r['errors']:
        ax4.semilogy(range(1,len(r['errors'])+1), r['errors'], 'o-', color=col, lw=1.5, ms=4, label=name)
ax4.set_title('Root Finding Convergence', fontweight='bold', fontsize=10)
ax4.set_xlabel('Iteration', fontsize=8); ax4.set_ylabel('Error (log)', fontsize=8); ax4.legend(fontsize=7)

# Panel 5: Euler vs RK4
ax5 = fig.add_subplot(3, 3, 5)
from simulation import euler_simulation
euler_S2 = euler_simulation(S0_C, rain_arr, et_arr, irr_zero, zc['field_capacity_pct'],
                              zc['drainage_coefficient'], zc['min_moisture_pct'])
dates_ext = pd.date_range('2026-03-01', periods=31, freq='D')
ax5.plot(dates_ext, euler_S2, 'b-', lw=1.5, label='Euler')
ax5.plot(dates_ext, rk_S,    'r--', lw=1.5, label='RK4')
ax5.axhline(zc['min_moisture_pct'], color='red', ls=':', lw=1, label='Stress')
ax5.set_title('Euler vs RK4 – Zone_C', fontweight='bold', fontsize=10)
ax5.set_ylabel('SM (%)', fontsize=8); ax5.legend(fontsize=7)
ax5.tick_params(axis='x', rotation=30, labelsize=7)

# Panel 6: Monte Carlo uncertainty
ax6 = fig.add_subplot(3, 3, 6)
summary = summarise_monte_carlo(mc_m)
ax6.fill_between(dates_ext, summary['p5'], summary['p95'], alpha=0.15, color='steelblue')
ax6.fill_between(dates_ext, summary['p25'], summary['p75'], alpha=0.3, color='steelblue')
ax6.plot(dates_ext, summary['mean'], color='steelblue', lw=2)
ax6.axhline(zc['min_moisture_pct'], color='red', ls=':', lw=1)
ax6.set_title(f'Monte Carlo (n=1000)\nShortage prob: {shortage_prob:.0f}%', fontweight='bold', fontsize=10)
ax6.set_ylabel('SM (%)', fontsize=8)
ax6.tick_params(axis='x', rotation=30, labelsize=7)

# Panel 7: Optimized schedule
ax7 = fig.add_subplot(3, 3, 7)
dates_30 = pd.date_range('2026-03-01', periods=30, freq='D')
ax7.bar(dates_30, opt['optimised_schedule'], color='#5aab61', alpha=0.8)
ax7.set_title(f"Optimized Irrigation Schedule\nTotal: {opt['total_water_used_mm']:.1f} mm", fontweight='bold', fontsize=10)
ax7.set_ylabel('Irrigation (mm)', fontsize=8)
ax7.tick_params(axis='x', rotation=30, labelsize=7)

# Panel 8: Optimization trajectory
ax8 = fig.add_subplot(3, 3, 8)
ax8.plot(dates_ext, opt['S_trajectory'], 'g-o', lw=2, ms=3, label='Optimized')
ax8.plot(dates_ext, rk_S, '--', color='grey', lw=1.5, alpha=0.7, label='No irrigation')
ax8.axhline(zc['min_moisture_pct'], color='red', ls=':', lw=1)
ax8.axhline(zc['target_moisture_pct'], color='green', ls=':', lw=1, alpha=0.6)
ax8.set_title('Zone_C with Optimal Irrigation', fontweight='bold', fontsize=10)
ax8.set_ylabel('SM (%)', fontsize=8); ax8.legend(fontsize=7)
ax8.tick_params(axis='x', rotation=30, labelsize=7)

# Panel 9: Key metrics table
ax9 = fig.add_subplot(3, 3, 9)
ax9.axis('off')
table_data = [
    ['Metric', 'Value'],
    ['Total rainfall (Mar)', f"{rain_arr.sum():.1f} mm"],
    ['Mean daily ET', f"{et_arr.mean():.2f} mm/day"],
    ['Zone_C stress days (no irr)', f"{np.sum(rk_S[1:] < zc['min_moisture_pct'])}"],
    ['Shortage probability (MC)', f"{shortage_prob:.0f}%"],
    ['Optimal total irrigation', f"{opt['total_water_used_mm']:.1f} mm"],
    ['Optimal stress days', f"{opt['stress_days']}"],
    ['Irrigation to target (root)', f"{r['root']:.2f} mm"],
    ['30-day water deficit', f"{total_deficit:.2f} mm-days"],
]
tbl = ax9.table(cellText=table_data[1:], colLabels=table_data[0],
                loc='center', cellLoc='left')
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5)
tbl.scale(1.2, 1.4)
ax9.set_title('Key Results Summary', fontweight='bold', fontsize=10)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig('../reports/fig11_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("Dashboard saved to reports/fig11_dashboard.png")

## 5. Reproducibility Checklist
| Item | Status |
|---|---|
| `README.md` with setup instructions | ✅ |
| `requirements.txt` with pinned libraries | ✅ |
| All notebooks run top-to-bottom without errors | ✅ |
| Random seeds fixed in Monte Carlo (`seed=42`) | ✅ |
| All data paths are relative (`../data/`) | ✅ |
| Raw datasets committed to `data/raw/` | ✅ |
| Cleaned dataset auto-generated by Level 4 | ✅ |
| `tests/` folder with ≥8 pytest tests | ✅ |
| `AI_USE_LOG.md` with 5 documented entries | ✅ |
| All figures saved to `reports/` | ✅ |

## 6. Final Conclusions

**Scientific conclusions:**
1. Zone_C (Maize, 180 m²) is the most water-stressed zone, approaching the 20% stress threshold in the final week of March without supplemental irrigation.
2. Mean daily evapotranspiration of ~4–5 mm/day is the dominant driver of moisture loss, exceeding average daily rainfall contribution for ~60% of March days.
3. Monte Carlo analysis confirms a non-negligible shortage probability, underscoring the need for a proactive irrigation plan rather than reactive response.
4. The gradient-descent optimized irrigation schedule eliminates stress days while using significantly less water than the greedy baseline — a meaningful saving for water-scarce smallholder contexts.

**Computational conclusions:**
1. NumPy vectorization delivers 10–50× speedup over Python loops with identical numerical results.
2. Newton-Raphson converges fastest for smooth differentiable functions; bisection is most robust as a fallback.
3. RK4 and Euler produce near-identical results for the daily time-step water balance, validating the simpler Euler approach for operational use.
4. All AI-generated code was validated against known analytical solutions, unit tests, and cross-checked with NumPy reference implementations.